In [1]:
import pandas as pd
import numpy as np

# Load validation predictions
val_df = pd.read_pickle('../data/ca1_val_predictions.pkl')

# Sort cleanly by SKU and date
val_df = val_df.sort_values(['id', 'date']).reset_index(drop=True)

# Assign realistic supply chain parameters per SKU
np.random.seed(42)
skus = val_df[['id', 'cat_id', 'sell_price']].drop_duplicates('id').copy()

# Assume wholesale unit cost is ~60% of retail sell_price
skus['unit_cost'] = skus['sell_price'] * 0.60
# Holding cost: ~25% annual rate -> ~0.07% per day per unit held
skus['daily_holding_cost'] = skus['unit_cost'] * (0.25 / 365)
# Stockout penalty: Lost gross margin (40% of sell_price) per unfulfilled unit
skus['stockout_cost'] = skus['sell_price'] * 0.40
# Supplier lead time (days): 3 days for FOODS, 5 days for HOBBIES/HOUSEHOLD
skus['lead_time'] = np.where(skus['cat_id'] == 'FOODS', 3, 5)

print(f"Configured supply chain parameters for {len(skus):,} SKUs.")
print(skus.head())


Configured supply chain parameters for 3,049 SKUs.
                              id cat_id  sell_price  unit_cost  \
0    FOODS_1_001_CA_1_evaluation  FOODS        2.24      1.344   
28   FOODS_1_002_CA_1_evaluation  FOODS        9.48      5.688   
56   FOODS_1_003_CA_1_evaluation  FOODS        3.23      1.938   
84   FOODS_1_004_CA_1_evaluation  FOODS        1.96      1.176   
112  FOODS_1_005_CA_1_evaluation  FOODS        3.54      2.124   

     daily_holding_cost  stockout_cost  lead_time  
0              0.000921          0.896          3  
28             0.003896          3.792          3  
56             0.001327          1.292          3  
84             0.000805          0.784          3  
112            0.001455          1.416          3  


In [3]:
def simulate_inventory(df_val, sku_params, forecast_col, service_level_z=1.65):
    """
    Simulates daily replenishment over the 28-day validation window.
    service_level_z = 1.65 corresponds to a 95% target cycle service level.
    """
    sim_df = df_val.merge(sku_params[['id', 'unit_cost', 'daily_holding_cost', 'stockout_cost', 'lead_time']], on='id')
    
    sim_df['rop'] = (
        sim_df[forecast_col] * sim_df['lead_time'] + 
        service_level_z * sim_df['rolling_std_28'].fillna(0) * np.sqrt(sim_df['lead_time'])
    )
    
    results = []
    for sku_id, group in sim_df.groupby('id', sort=False):
        actuals = group['sales'].to_numpy()
        forecasts = group[forecast_col].to_numpy()
        rops = group['rop'].to_numpy()
        lt = int(group['lead_time'].iloc[0])
        h_cost = float(group['daily_holding_cost'].iloc[0])
        s_cost = float(group['stockout_cost'].iloc[0])
        
        # Start Day 1 with inventory equal to initial ROP (scalar float)
        on_hand = float(rops[0])
        pipeline = []  # list of (arrival_day, order_qty)
        
        total_holding = 0.0
        total_stockout = 0.0
        lost_units = 0.0
        total_demand = float(actuals.sum())
        
        for day in range(len(actuals)):
            # 1. Receive arriving orders
            arriving = sum(qty for arr_day, qty in pipeline if arr_day == day)
            on_hand += arriving
            
            # 2. Fulfill today's actual customer demand
            demand = float(actuals[day])
            if on_hand >= demand:
                on_hand -= demand
            else:
                shortfall = demand - on_hand
                lost_units += shortfall
                total_stockout += shortfall * s_cost
                on_hand = 0.0
            
            # 3. Accrue end-of-day holding cost
            total_holding += on_hand * h_cost
            
            # 4. Check Inventory Position (On-Hand + On-Order) vs ROP
            on_order = sum(qty for arr_day, qty in pipeline if arr_day > day)
            inv_position = on_hand + on_order
            
            if inv_position <= rops[day]:
                order_qty = max(0.0, (forecasts[day] * (lt + 7)) - inv_position)
                if order_qty > 0:
                    pipeline.append((day + lt, order_qty))
                    
        results.append({
            'id': sku_id,
            'total_demand': total_demand,
            'lost_units': lost_units,
            'holding_cost': total_holding,
            'stockout_cost': total_stockout,
            'total_cost': total_holding + total_stockout
        })
        
    return pd.DataFrame(results)

# Run simulation for both policies (at 95% target service level, Z=1.65)
res_baseline = simulate_inventory(val_df, skus, forecast_col='rolling_mean_28', service_level_z=1.65)
res_lgb = simulate_inventory(val_df, skus, forecast_col='pred_lgb', service_level_z=1.65)

# Compare Total Business Impact across Store CA_1 (28 Days)
summary = pd.DataFrame({
    'Metric': ['Total Demand (Units)', 'Lost Sales (Units)', 'Fill Rate (%)', 'Holding Cost ($)', 'Stockout Penalty ($)', 'Total Inventory Cost ($)'],
    'Baseline (28-Day MA)': [
        res_baseline['total_demand'].sum(),
        res_baseline['lost_units'].sum(),
        (1 - res_baseline['lost_units'].sum() / res_baseline['total_demand'].sum()) * 100,
        res_baseline['holding_cost'].sum(),
        res_baseline['stockout_cost'].sum(),
        res_baseline['total_cost'].sum()
    ],
    'LightGBM Forecast': [
        res_lgb['total_demand'].sum(),
        res_lgb['lost_units'].sum(),
        (1 - res_lgb['lost_units'].sum() / res_lgb['total_demand'].sum()) * 100,
        res_lgb['holding_cost'].sum(),
        res_lgb['stockout_cost'].sum(),
        res_lgb['total_cost'].sum()
    ]
})

summary['Difference (LGB - Base)'] = summary['LightGBM Forecast'] - summary['Baseline (28-Day MA)']
print(summary.to_string(index=False, float_format=lambda x: f"{x:,.2f}"))

# Save simulation results for the Streamlit Dashboard
res_baseline.to_pickle('../data/sim_baseline.pkl')
res_lgb.to_pickle('../data/sim_lgb.pkl')
print("\nSaved simulation results successfully!")


                  Metric  Baseline (28-Day MA)  LightGBM Forecast  Difference (LGB - Base)
    Total Demand (Units)            134,032.00         134,032.00                     0.00
      Lost Sales (Units)             13,136.97           8,998.26                -4,138.71
           Fill Rate (%)                 90.20              93.29                     3.09
        Holding Cost ($)                974.94           1,064.59                    89.66
    Stockout Penalty ($)             17,986.37          11,605.37                -6,381.00
Total Inventory Cost ($)             18,961.31          12,669.97                -6,291.34

Saved simulation results successfully!
